# Proyecto final · Cálculo Integral 2026-II
## Una tasa, una pregunta y una cantidad acumulada

**Equipo:** (nombres) · **Conjunto de datos:** (D01 a D12)

Este cuaderno es un punto de partida. Trae la carga de los datos, tres métodos de integración numérica explicados y la celda de ajuste. Lo demás lo escriben ustedes, en el orden de las piezas P1 a P6 de la guía.

Dos reglas: **toda decisión se escribe y se defiende** (bitácora de decisiones), y cualquier integrante debe poder explicar cualquier celda, incluidas las que ya vienen escritas.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sympy as sp
from scipy.optimize import curve_fit

---
## P1. Los datos y la tasa

Cambien `ARCHIVO` por el nombre de su conjunto (está en su ficha).

In [ ]:
BASE = "https://raw.githubusercontent.com/cmorregof/labs-calculo-2026-2/main/integral/proyecto_final/datos/"
ARCHIVO = "D01_chinchina_creciente_2026.csv"      # <- cambien esta línea

datos = pd.read_csv(BASE + ARCHIVO)
datos.head()

In [ ]:
col_t = [c for c in datos.columns if c.startswith("t_")][0]   # la columna del tiempo
col_r = datos.columns[-1]                                      # la columna de la tasa
print("Tiempo:", col_t, "| Tasa:", col_r)
print("Filas:", len(datos), "| Celdas vacías en la tasa:", datos[col_r].isna().sum())

**Escriban aquí, antes de seguir:**

- Unidades de la tasa:
- Unidades del tiempo:
- Unidades del producto tasa × tiempo, y a qué unidad lo van a convertir:
- Datos faltantes o irregulares que encontraron y qué decidieron hacer con ellos (también va a la bitácora de decisiones):

In [ ]:
# Aquí va su decisión sobre los datos faltantes. Al terminar, t y r no deben tener vacíos.
limpio = datos.dropna(subset=[col_r])       # <- esta es UNA opción. ¿Es la suya? Defiéndanla o cámbienla.
t = limpio[col_t].to_numpy(dtype=float)
r = limpio[col_r].to_numpy(dtype=float)

plt.figure(figsize=(9, 3.5))
plt.plot(t, r, "o-", ms=3)
plt.xlabel(col_t); plt.ylabel(col_r); plt.title(ARCHIVO); plt.grid(alpha=.3)
plt.show()

---
## P2. La predicción

Impriman la gráfica anterior (o dibújenla en papel). **Sin calcular nada**, estimen a ojo la cantidad acumulada con rectángulos o triángulos dibujados a mano. Tomen la foto e insértenla aquí.

- Nuestra predicción (con unidades):

---
## P3. La acumulación numérica

Queremos $\displaystyle\int_{t_0}^{t_n} r(t)\,dt$, pero no tenemos una fórmula para $r$: tenemos los valores $r_0, r_1, \dots, r_n$ medidos en los instantes $t_0 < t_1 < \dots < t_n$. Los tres métodos de abajo responden la misma pregunta de tres maneras: **¿qué hacemos con la tasa entre un dato y el siguiente?** Antes de escoger, lean en su ficha qué es cada dato: ¿el valor en un instante o el promedio de un intervalo?

### Método 1: suma de Riemann

Suponemos que la tasa se queda **constante** entre un dato y el siguiente, como en las funciones escalonadas de L1. Con el extremo izquierdo de cada subintervalo:

$$\int_{t_0}^{t_n} r(t)\,dt \;\approx\; \sum_{k=0}^{n-1} r_k\,(t_{k+1}-t_k).$$

Cada sumando es un rectángulo: tasa por lo que dura. Con el extremo derecho se usa $r_{k+1}$ en lugar de $r_k$.

In [ ]:
def riemann_izquierda(t, r):
    anchos = np.diff(t)              # t[k+1] - t[k]
    return np.sum(r[:-1] * anchos)   # altura: el dato de la izquierda

def riemann_derecha(t, r):
    anchos = np.diff(t)
    return np.sum(r[1:] * anchos)    # altura: el dato de la derecha

### Método 2: regla del trapecio

Suponemos que entre dos datos la tasa cambia **en línea recta**. La región bajo ese segmento es un trapecio, y su área es el ancho por el promedio de las dos alturas:

$$\int_{t_k}^{t_{k+1}} r(t)\,dt \;\approx\; \frac{r_k + r_{k+1}}{2}\,(t_{k+1}-t_k),
\qquad\text{así que}\qquad
\int_{t_0}^{t_n} r(t)\,dt \;\approx\; \sum_{k=0}^{n-1} \frac{r_k + r_{k+1}}{2}\,(t_{k+1}-t_k).$$

Miren la fórmula con calma: el trapecio es el **promedio** de la suma de Riemann izquierda y la derecha. Si la tasa es cóncava hacia abajo el trapecio subestima, y si es cóncava hacia arriba, sobreestima. ¿Por qué?

In [ ]:
def trapecio(t, r):
    anchos = np.diff(t)
    alturas_promedio = (r[:-1] + r[1:]) / 2
    return np.sum(alturas_promedio * anchos)

### Método 3: regla de Simpson

Ahora tomamos los datos de a tres y suponemos que por esos tres puntos pasa una **parábola**. Si los datos están igualmente espaciados, con paso $h$, el área bajo la parábola que pasa por $(t_0,r_0)$, $(t_1,r_1)$ y $(t_2,r_2)$ es

$$\int_{t_0}^{t_2} r(t)\,dt \;\approx\; \frac{h}{3}\,\bigl(r_0 + 4r_1 + r_2\bigr).$$

Al pegar los tramos de a dos subintervalos queda

$$\int_{t_0}^{t_n} r(t)\,dt \;\approx\; \frac{h}{3}\,\bigl(r_0 + 4r_1 + 2r_2 + 4r_3 + \dots + 2r_{n-2} + 4r_{n-1} + r_n\bigr).$$

Simpson pide dos cosas: **paso constante** y un **número par de subintervalos** (es decir, un número impar de datos). Si sus datos no las cumplen, la función de abajo se detiene y les dice por qué. Qué hacer entonces es una decisión del equipo.

In [ ]:
def simpson(t, r):
    h = np.diff(t)
    if not np.allclose(h, h[0]):
        raise ValueError("Simpson pide paso constante y aquí los tiempos no están igualmente espaciados.")
    n = len(t) - 1                       # número de subintervalos
    if n % 2 == 1:
        raise ValueError(f"Simpson pide un número par de subintervalos y aquí hay {n}.")
    return h[0] / 3 * (r[0] + 4 * np.sum(r[1:-1:2]) + 2 * np.sum(r[2:-1:2]) + r[-1])

### Su turno

Calculen la integral con **al menos dos métodos o dos pasos**. Para cambiar el paso pueden usar uno de cada dos datos: `t[::2], r[::2]`. Conviertan el resultado a una unidad que se entienda y expliquen **por qué** los resultados difieren.

In [ ]:
# Su código aquí. Por ejemplo:
# print(riemann_izquierda(t, r), trapecio(t, r))

### La función de acumulación

$A(t_k)=\displaystyle\int_{t_0}^{t_k} r(s)\,ds$ es la cantidad acumulada **hasta** cada instante. Es la misma idea de la curva del contador de L1 y de $D(x)$ en L2. Aquí se calcula sumando trapecios uno tras otro.

In [ ]:
def acumulada(t, r):
    trozos = (r[:-1] + r[1:]) / 2 * np.diff(t)
    return np.concatenate([[0.0], np.cumsum(trozos)])

# A = acumulada(t, r)
# plt.plot(t, A); plt.xlabel(col_t); plt.ylabel("acumulado (¿en qué unidades?)"); plt.show()

**Confronten con P2.** ¿Cuánto se alejó la predicción a ojo del resultado? ¿Por qué?

---
## P4. El modelo

Ajusten una función a la tasa usando **solo los primeros dos tercios** de los datos. El último tercio se guarda para comparar en P5.

Deben probar **al menos dos familias** y elegir una. La función elegida debe pedir una técnica del curso al integrarla (sustitución, partes, fracciones parciales o sustitución trigonométrica).

`curve_fit` busca los valores de los parámetros que hacen más pequeña la suma de los cuadrados de las distancias entre los datos y la función. Necesita un punto de partida, `p0`: si está muy lejos, el ajuste puede fallar.

In [ ]:
corte = 2 * len(t) // 3
t_ajuste, r_ajuste = t[:corte], r[:corte]        # lo que el modelo ve
t_prueba, r_prueba = t[corte - 1:], r[corte - 1:]  # lo que el modelo NO ve (empieza donde termina el ajuste)

def modelo(t, a, b):
    return a + b * t          # <- reemplacen por su familia. Esta recta es solo un ejemplo y no cumple el requisito.

p0 = [r_ajuste.mean(), 0.0]   # <- un valor inicial razonable para cada parámetro
parametros, _ = curve_fit(modelo, t_ajuste, r_ajuste, p0=p0, maxfev=20000)
print("Parámetros:", parametros)

error = np.sqrt(np.mean((r_ajuste - modelo(t_ajuste, *parametros)) ** 2))
print("Error típico del ajuste (en las unidades de la tasa):", error)

plt.figure(figsize=(9, 3.5))
plt.plot(t, r, "o", ms=3, label="datos")
tt = np.linspace(t[0], t[-1], 400)
plt.plot(tt, modelo(tt, *parametros), label="modelo")
plt.axvline(t[corte - 1], ls="--", c="gray"); plt.legend(); plt.grid(alpha=.3)
plt.show()

**Escriban aquí:** las dos familias que probaron, cuál eligieron y por qué (el error **y** un argumento sobre el fenómeno), y qué significa cada parámetro con sus unidades.

---
## P5. La integral a mano y la comparación

1. Integren su modelo **a mano**, con el procedimiento completo. Inserten la foto o escríbanlo en el informe.
2. Verifiquen con SymPy (celda de abajo).
3. Con su antiderivada, calculen lo acumulado en el **último tercio** y compárenlo con la integral numérica de ese mismo tramo.

In [ ]:
x, a, b = sp.symbols("x a b", real=True)
f = a + b * x                      # <- escriban aquí su modelo con símbolos
F_sympy = sp.integrate(f, x)       # lo que dice SymPy
F_mia = a * x + b * x**2 / 2       # <- escriban aquí SU antiderivada, la que hicieron a mano
print("SymPy:", F_sympy)
print("¿Mi antiderivada derivada devuelve el modelo?", sp.simplify(sp.diff(F_mia, x) - f) == 0)

In [ ]:
# Comparación en el último tercio. Su código aquí:
# numerico = trapecio(t_prueba, r_prueba)
# analitico = F(t_prueba[-1]) - F(t_prueba[0])     # con su antiderivada y sus parámetros
# print(numerico, analitico, 100 * (analitico - numerico) / numerico, "%")

**Escriban aquí:** cuánto difieren y **por qué**. ¿El modelo se equivoca hacia arriba o hacia abajo? ¿Qué pasó en el último tercio que el modelo no podía saber?

---
## P6. La respuesta y sus límites

- Respuesta a la pregunta de su ficha, como un **rango** con unidades:
- Qué parte del trabajo es dato y qué parte es supuesto:
- Cambien una de sus decisiones (otro método, otro trato a los faltantes, la otra familia) y recalculen. ¿Cuánto se mueve la respuesta?

---
## Extensión optativa

Si su ficha indica una extensión y quieren hacerla, va aquí.

---
## Antes de entregar

`Entorno de ejecución → Reiniciar y ejecutar todo`. El cuaderno debe correr de arriba a abajo sin errores.